In [ ]:
import numpy as np
import pandas as pd
import random
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, LSTM, Dense, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.datasets import imdb

positive_sentences = [
    "I love this product", "This movie was amazing", "The service was excellent",
    "I am very happy", "Fantastic experience", "Great quality", "Absolutely wonderful",
    "I enjoyed it", "Highly recommended", "Best purchase", "Very satisfying",
    "This made my day", "Superb performance", "I really like it", "Outstanding work",
    "It works perfectly", "Great support", "Very impressive", "Loved it", "Worth the money"
]

negative_sentences = [
    "I hate this product", "This movie was terrible", "The service was awful",
    "I am very disappointed", "Worst experience", "Poor quality", "Absolutely horrible",
    "I regret buying this", "Not recommended", "Waste of money", "Very unsatisfying",
    "This ruined my day", "Bad performance", "I dislike it", "Terrible work",
    "It does not work", "Very bad support", "Not impressive", "I hated it", "Totally useless"
]

data = []
for _ in range(300):
    data.append([random.choice(positive_sentences), 1])
    data.append([random.choice(negative_sentences), 0])

df = pd.DataFrame(data, columns=["text", "sentiment"])

tokenizer = Tokenizer(num_words=1000)
tokenizer.fit_on_texts(df["text"])
sequences = tokenizer.texts_to_sequences(df["text"])

X_dummy = pad_sequences(sequences, maxlen=10)
y_dummy = df["sentiment"].values

X_train_dummy, X_test_dummy, y_train_dummy, y_test_dummy = train_test_split(
    X_dummy, y_dummy, test_size=0.25, random_state=42
)

# 2. RNN on Dummy Data

In [ ]:
rnn_dummy = Sequential([
    Embedding(input_dim=1000, output_dim=32),
    SimpleRNN(64, activation="tanh"),
    Dense(1, activation="sigmoid")
])

rnn_dummy.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
rnn_dummy.fit(X_train_dummy, y_train_dummy, epochs=2, validation_data=(X_test_dummy, y_test_dummy))

loss, acc_rnn_dummy = rnn_dummy.evaluate(X_test_dummy, y_test_dummy)
print("Simple RNN Accuracy on Dummy Data:", acc_rnn_dummy)

# 3. LSTM on Dummy Data

In [ ]:
lstm_dummy = Sequential([
    Embedding(input_dim=1000, output_dim=32),
    LSTM(64),
    Dense(1, activation="sigmoid")
])

lstm_dummy.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
lstm_dummy.fit(X_train_dummy, y_train_dummy, epochs=2, validation_data=(X_test_dummy, y_test_dummy))

loss, acc_lstm_dummy = lstm_dummy.evaluate(X_test_dummy, y_test_dummy)
print("LSTM Accuracy on Dummy Data:", acc_lstm_dummy)

# 4. Dataset Loading (IMDB)

In [ ]:
max_features = 10000

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_features)

x_train = pad_sequences(x_train)
x_test = pad_sequences(x_test)

# 5. RNN on IMDB Dataset

In [ ]:
rnn_model = Sequential([
    Embedding(input_dim=max_features, output_dim=32),
    SimpleRNN(64, activation='tanh'),
    Dense(1, activation='sigmoid')
])

rnn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

rnn_model.fit(x_train[:2000], y_train[:2000], epochs=3, batch_size=64, validation_split=0.2)

loss, acc_rnn_imdb = rnn_model.evaluate(x_test[:500], y_test[:500])
print("Simple RNN Accuracy on IMDB Dataset:", acc_rnn_imdb)

# 6. LSTM on IMDB Dataset

In [ ]:
lstm_model = Sequential([
    Embedding(input_dim=max_features, output_dim=32),
    LSTM(64),
    Dense(1, activation='sigmoid')
])

lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

lstm_model.fit(x_train[:2000], y_train[:2000], epochs=3, batch_size=64, validation_split=0.2)

loss, acc_lstm_imdb = lstm_model.evaluate(x_test[:500], y_test[:500])
print("LSTM Accuracy on IMDB Dataset:", acc_lstm_imdb)

# 7. Comparison of RNN and LSTM

In [ ]:
print("=== Final Comparison")
print("Dummy Data -> RNN:", acc_rnn_dummy, "| LSTM:", acc_lstm_dummy)
print("IMDB Data  -> RNN:", acc_rnn_imdb, "| LSTM:", acc_lstm_imdb)